In [ ]:
# ---------------------------------------------------------------
# CAPÍTULO 12 — Ejercicio 1
# ---------------------------------------------------------------

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X_dig, _ = load_digits(return_X_y=True)
X_sc = StandardScaler().fit_transform(X_dig)

pca_full = PCA().fit(X_sc)
varianza_acum = np.cumsum(pca_full.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(1, len(varianza_acum)+1), varianza_acum)

for umbral, color in [(0.90, "green"), (0.95, "orange"), (0.99, "red")]:
    n_comp = np.searchsorted(varianza_acum, umbral) + 1
    ax.axhline(umbral, color=color, linestyle="--", label=f"{int(umbral*100)}% → {n_comp} comps")
    ax.axvline(n_comp, color=color, linestyle=":")

var_acum = np.cumsum(pca_full.explained_variance_ratio_)

# Cálculo de componentes para cada umbral
n_90 = np.argmax(var_acum >= 0.90) + 1
n_95 = np.argmax(var_acum >= 0.95) + 1
n_99 = np.argmax(var_acum >= 0.99) + 1

print(f'Componentes para 90% varianza: {n_90} (se reducen {64 - n_90})')
print(f'Componentes para 95% varianza: {n_95} (se reducen {64 - n_95})')
print(f'Componentes para 99% varianza: {n_99} (se reducen {64 - n_99})')

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(var_acum)+1), var_acum, 'o-', ms=3)
plt.axhline(0.90, color='green', linestyle='--', label='90%')
plt.axhline(0.95, color='red', linestyle='--', label='95%')
plt.axhline(0.99, color='orange', linestyle='--', label='99%')
plt.axvline(n_90, color='green', linestyle=':', alpha=0.5)
plt.axvline(n_95, color='red', linestyle=':', alpha=0.5)
plt.axvline(n_99, color='orange', linestyle=':', alpha=0.5)
plt.xlabel('Número de componentes')
plt.ylabel('Varianza explicada acumulada')
plt.title('Varianza explicada acumulada — load_digits')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ---------------------------------------------------------------
# CAPÍTULO 12 — Ejercicio 2
# ---------------------------------------------------------------

from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt

n_comps = [2, 5, 10, 20, 30, 50, 64]
cv_means = []

for n in n_comps:
    pipe_pca = Pipeline([
        ("pca", PCA(n_components=n)),
        ("knn", KNeighborsClassifier(n_neighbors=5)),
    ])
    scores = cross_val_score(pipe_pca, X_sc, _, cv=5, n_jobs=-1)
    cv_means.append(scores.mean())
    print(f"n={n:3d} | accuracy={scores.mean():.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(n_comps, cv_means, marker="o")
ax.set_xlabel("n_components")
ax.set_ylabel("Accuracy (CV=5)")
ax.set_title("PCA + KNN — Accuracy vs n_components")
plt.tight_layout()
plt.show()


In [ ]:
# ---------------------------------------------------------------
# CAPÍTULO 12 — Ejercicio 3
# ---------------------------------------------------------------

from sklearn.datasets import fetch_openml
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np

try:
    mnist = fetch_openml("mnist_784", version=1, as_frame=False, parser="auto")
    X_mn = mnist.data[:5000] / 255.0
    y_mn = mnist.target[:5000].astype(int)
except Exception as e:
    from sklearn.datasets import load_digits
    X_mn, y_mn = load_digits(return_X_y=True)
    X_mn = StandardScaler().fit_transform(X_mn)

# PCA a 2 componentes
X_pca2 = PCA(n_components=2, random_state=42).fit_transform(X_mn)

# t-SNE con PCA previo a 50 componentes
X_pca50 = PCA(n_components=50, random_state=42).fit_transform(X_mn)
X_tsne  = TSNE(n_components=2, perplexity=30, learning_rate="auto", random_state=42).fit_transform(X_pca50)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, X_2d, titulo in zip(
    axes,
    [X_pca2, X_tsne],
    ["PCA (2 componentes)", "t-SNE (perplexity=30)"]
):
    sc = ax.scatter(X_2d[:, 0], X_2d[:, 1],
                    c=y_mn, cmap="tab10", s=5, alpha=0.7)
    ax.set_title(titulo)
    ax.axis("off")
plt.colorbar(sc, ax=axes, label="Dígito")
plt.suptitle("PCA vs t-SNE — MNIST")
plt.tight_layout()
plt.show()
